Testing the model

Setting up the MIDI input 

In [8]:
import mido





In [9]:
print("Input ports:", mido.get_input_names())
print("Output ports:", mido.get_output_names())

Input ports: ['Digital Keyboard 0', 'PythonOut 1', 'PythonOut 2 2']
Output ports: ['Microsoft GS Wavetable Synth 0', 'Digital Keyboard 1', 'PythonOut 2', 'PythonOut 2 3']


In [10]:
keyboard_port = 'Digital Keyboard 0'
virtual_port = 'PythonOut 2'  # This is the output port to Cakewalk

keyboard_in = mido.open_input(keyboard_port)
virtual_out = mido.open_output(virtual_port)

print(f"Forwarding MIDI from '{keyboard_port}' to '{virtual_port}'... (Ctrl+C to stop)")

try:
    for msg in keyboard_in:
        print(msg)
        virtual_out.send(msg)
except KeyboardInterrupt:
    print("Stopped MIDI forwarding.")
finally:
    keyboard_in.close()
    virtual_out.close()

Forwarding MIDI from 'Digital Keyboard 0' to 'PythonOut 2'... (Ctrl+C to stop)
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=0
clock time=

In [ ]:
import mido
import time
from mido import Message

output_port = 'PythonOut 2'
outport = mido.open_output(output_port)

# --- Simple parser for REMI-style tokens ---
current_velocity = 100  # Default velocity
note_length = 0.4       # Seconds to hold each note

for token in generated_tokens:
    if token.startswith("<Velocity_"):
        # Update velocity for next notes
        try:
            current_velocity = int(token.replace("<Velocity_", "").replace(">", ""))
        except:
            current_velocity = 100
    elif token.startswith("<Note_ON_"):
        try:
            note = int(token.replace("<Note_ON_", "").replace(">", ""))
            # Send note_on
            outport.send(Message('note_on', note=note, velocity=current_velocity, time=0))
            time.sleep(note_length)
            # Send note_off
            outport.send(Message('note_off', note=note, velocity=current_velocity, time=0))
        except:
            pass
    # You can add more parsing for channels, instruments, etc. if your tokens include them

outport.close()
print("Done playing generated sequence!")

In [6]:
print("Input ports:", mido.get_input_names())
print("Output ports:", mido.get_output_names())

Input ports: ['Digital Keyboard 0', 'PythonOut 1', 'PythonOut 2 2']
Output ports: ['Microsoft GS Wavetable Synth 0', 'Digital Keyboard 1', 'PythonOut 2', 'PythonOut 2 3']


In [4]:
import torch
import json
import mido
from mido import MidiFile, Message, open_input, open_output
from pathlib import Path

# === CONFIG ===
project_root = Path.cwd().parent
model_path = project_root / "music_lstm_model.pth" # Path to your saved model
vocab_path = project_root / "vocab.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === LOAD VOCAB ===
with open(vocab_path) as f:
    vocab = json.load(f)
stoi = vocab
itos = {i: t for t, i in vocab.items()}

# === LOAD MODEL ===
# Replace with your actual model class if needed
class LSTMModel(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, num_layers=2):
        super().__init__()
        self.embed = torch.nn.Embedding(vocab_size, embed_dim)
        self.lstm = torch.nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embed(x)
        x, hidden = self.lstm(x, hidden)
        x = self.fc(x)
        return x, hidden

model = LSTMModel(vocab_size=len(vocab))
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# === MIDI INPUT/OUTPUT ===
input_name = 'Digital Keyboard 0'  # Or select from list
output_name = 'PythonOut 2'  # Your DAW or synth input

inport = open_input(input_name)
outport = open_output(output_name)

# === JAM LOOP ===
context = []  # Keep last N tokens
hidden = None

print("🎹 Ready to jam! Play on your MIDI keyboard...")

for msg in inport:
    if msg.type == "note_on" and msg.velocity > 0:
        pitch = msg.note
        velocity = msg.velocity

        # Convert to tokens
        tokens = [
            "<Position_0>",  # Simplified — adjust as needed
            f"<Note_ON_{pitch}>",
            f"<Velocity_{velocity}>"
        ]

        # Update context
        for t in tokens:
            context.append(stoi.get(t, stoi["<UNK>"]))
        context = context[-128:]  # Keep recent tokens only

        # Predict next token
        input_tensor = torch.tensor(context).unsqueeze(0).to(device)
        with torch.no_grad():
            output, hidden = model(input_tensor, hidden)
            prediction = output[0, -1].argmax().item()
            predicted_token = itos.get(prediction, "<UNK>")

        print("🎶 Predicted:", predicted_token)

        # If prediction is a note, send it out
        if predicted_token.startswith("<Note_ON_"):
            pred_pitch = int(predicted_token.strip("<Note_ON_>"))
            out_msg = Message("note_on", note=pred_pitch, velocity=100, time=0)
            outport.send(out_msg)

C:\Users\Andrew Robinson\AppData\Local\Temp\ipykernel_13856\2126736253.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_

SystemError: MidiInWinMM::openPort: error creating Windows MM MIDI input port.

In [10]:
from mido import Message

# Example: send note_on for bass on channel 2 (channel numbers are 0-based)
out_msg = Message("note_on", note=48, velocity=100, time=0, channel=1)  # channel=5 is MIDI channel 2
outport.send(out_msg)

In [4]:
print("Output ports:", mido.get_output_names())
virtual_port = 'PythonOut 2'  # Use the exact name from above

# Only open if the port exists
if virtual_port in mido.get_output_names():
    virtual_out = mido.open_output(virtual_port)
else:
    raise RuntimeError(f"Output port '{virtual_port}' not found.")

Output ports: ['Microsoft GS Wavetable Synth 0', 'Digital Keyboard 1', 'PythonOut 2', 'PythonOut 2 3']


SystemError: MidiOutWinMM::openPort: error creating Windows MM MIDI output port.